In [4]:
# ==========================================
# RECRUITERVISION AI
# SKILL RECOMMENDATION MODEL TRAINING
# ==========================================

import pandas as pd
import re
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

# ==========================================
# LOAD DATASETS
# ==========================================

resume_df = pd.read_csv("/home/vidhi/Desktop/Python PGMS/RecruiterVisionAI/dataset/job_title_des.csv")
jd_df = pd.read_csv("/home/vidhi/Desktop/Python PGMS/RecruiterVisionAI/dataset/UpdatedResumeDataSet.csv")

# ==========================================
# SKILL DATABASE
# ==========================================

SKILLS_DB = [
    "python","java","c++","tensorflow","pytorch",
    "machine learning","deep learning","nlp",
    "sql","mongodb","flask","django","react",
    "node.js","aws","docker","kubernetes",
    "figma","ui ux","adobe xd","javascript",
    "typescript","data analysis","pandas",
    "numpy","scikit-learn","opencv","fastapi"
]

# ==========================================
# CLEAN TEXT
# ==========================================

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

# ==========================================
# EXTRACT SKILLS
# ==========================================

def extract_skills(text):

    text = clean_text(text)

    found = []

    for skill in SKILLS_DB:

        if skill.lower() in text:
            found.append(skill)

    return list(set(found))

# ==========================================
# CREATE TRAINING DATA
# ==========================================

X = []
y = []

# ==========================================
# RESUME COLUMN
# ==========================================

resume_col = resume_df.columns[0]

# ==========================================
# JOB DESCRIPTION COLUMN
# ==========================================

jd_col = jd_df.columns[0]

# ==========================================
# CREATE DATA PAIRS
# ==========================================

for i in range(min(len(resume_df), len(jd_df))):

    resume_text = str(resume_df[resume_col].iloc[i])

    jd_text = str(jd_df[jd_col].iloc[i])

    # Extract skills

    resume_skills = extract_skills(resume_text)

    jd_skills = extract_skills(jd_text)

    # Missing skills = recommendations

    missing_skills = list(
        set(jd_skills) - set(resume_skills)
    )

    # Skip empty labels

    if len(missing_skills) == 0:
        continue

    # Combine resume + JD

    combined_text = (
        clean_text(resume_text)
        + " "
        + clean_text(jd_text)
    )

    X.append(combined_text)

    y.append(missing_skills)

# ==========================================
# TF-IDF
# ==========================================

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_vectors = vectorizer.fit_transform(X)

# ==========================================
# MULTI LABEL ENCODER
# ==========================================

mlb = MultiLabelBinarizer()

y_encoded = mlb.fit_transform(y)

# ==========================================
# TRAIN TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X_vectors,
    y_encoded,
    test_size=0.2,
    random_state=42
)

# ==========================================
# MODEL
# ==========================================

model = MLPClassifier(

    hidden_layer_sizes=(256,128),

    activation='relu',

    solver='adam',

    max_iter=100,

    random_state=42,

    early_stopping=True,

    verbose=True
)

# ==========================================
# TRAIN
# ==========================================

print("\nTraining Started...\n")

model.fit(X_train, y_train)

print("\nModel Training Completed 😎🔥\n")

# ==========================================
# ACCURACY
# ==========================================

accuracy = model.score(X_test, y_test)

print(f"Model Accuracy: {accuracy * 100:.2f}%")

# ==========================================
# SAVE MODEL
# ==========================================

with open("skill_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("mlb_encoder.pkl", "wb") as f:
    pickle.dump(mlb, f)

print("\nAll files saved successfully 😎🔥")


Training Started...

Iteration 1, loss = 1.41956408
Validation score: 0.272727
Iteration 2, loss = 1.39247439
Validation score: 0.363636
Iteration 3, loss = 1.36748901
Validation score: 0.363636
Iteration 4, loss = 1.34409294
Validation score: 0.454545
Iteration 5, loss = 1.32201077
Validation score: 0.727273
Iteration 6, loss = 1.30116225
Validation score: 0.727273
Iteration 7, loss = 1.28071812
Validation score: 0.727273
Iteration 8, loss = 1.26012377
Validation score: 0.727273
Iteration 9, loss = 1.23890120
Validation score: 0.727273
Iteration 10, loss = 1.21674701
Validation score: 0.727273
Iteration 11, loss = 1.19350266
Validation score: 0.727273
Iteration 12, loss = 1.16896391
Validation score: 0.727273
Iteration 13, loss = 1.14280300
Validation score: 0.727273
Iteration 14, loss = 1.11509696
Validation score: 0.727273
Iteration 15, loss = 1.08570382
Validation score: 0.727273
Iteration 16, loss = 1.05454625
Validation score: 0.727273
Validation score did not improve more than 

In [9]:
import pickle
import re
import numpy as np

# ==========================================
# LOAD MODEL FILES
# ==========================================

with open("skill_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

with open("mlb_encoder.pkl", "rb") as f:
    mlb = pickle.load(f)

# ==========================================
# CLEAN TEXT
# ==========================================

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

# ==========================================
# SAMPLE INPUT
# ==========================================

resume_text = """
Python developer skilled in Flask,
SQL and REST APIs
"""

job_description = """
Looking for ML Engineer with
TensorFlow, NLP, Deep Learning,
PyTorch and AWS experience
"""

# ==========================================
# COMBINE TEXT
# ==========================================

combined = (
    clean_text(resume_text)
    + " " +
    clean_text(job_description)
)

# ==========================================
# VECTORIZE
# ==========================================

vector = vectorizer.transform([combined])

# ==========================================
# PREDICT
# ==========================================

prediction = model.predict(vector)

# ==========================================
# CONVERT LABELS
# ==========================================

recommended_skills = mlb.inverse_transform(prediction)

# ==========================================
# OUTPUT
# ==========================================

print("\n========== Recommended Skills ==========\n")

if len(recommended_skills[0]) == 0:

    print("❌ No skills predicted")

else:

    for skill in recommended_skills[0]:

        print("✅", skill)


========== Recommended Skills ==========

❌ No skills predicted
